# Env39 — Load and Validate the Canonical Street Network

This notebook teaches how a successfully published canonical GeoPackage becomes the project's two validated graph representations. The canonical network is the analysis source of truth. Its coordinates are local Cartesian; physical streets are undirected, while the OSMnx representation contains reciprocal directed arcs. Package functions perform all scientific validation and construction.

## 1. Configuration

Normally only this cell needs editing. `latest` selects the latest successful Phase 7C run; a specific published run ID may be supplied for reproduction.

In [ ]:
ENVIRONMENT = "env39"
CANONICAL_RUN = "latest"

## 2. Locate the repository and import the tested implementation

The discovery logic supports launching Jupyter Lab from either the repository root or `notebooks/`. It fails clearly instead of embedding a workstation-specific absolute path.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = (start, *start.parents)
PROJECT_ROOT = next(
    (candidate for candidate in candidates
     if (candidate / "environment.yml").is_file()
     and (candidate / "src" / "geogami_morphology").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Repository root not found. Start Jupyter from the repository or notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Repository root discovered:", PROJECT_ROOT.name)
print("REPOSITORY_DISCOVERY: PASS")

In [ ]:
from importlib.metadata import version
import json
import platform

import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import pyproj
import shapely
from IPython.display import SVG, display

from geogami_morphology.graph import (
    build_networkx_multigraph,
    build_osmnx_multidigraph,
    directed_arc_count,
    load_analysis_graphml,
    load_canonical_geopackage,
    physical_street_count,
    resolve_canonical_run,
    validate_canonical_for_analysis,
    validate_graph_correspondence,
    validate_graphml_roundtrip,
)
from geogami_morphology.io import sha256_file
from geogami_morphology.notebook import canonical_network_svg

## 3. Software versions

Recording the active software makes exploratory work interpretable and supports later reproduction.

In [ ]:
packages = ["osmnx", "networkx", "geopandas", "shapely", "pandas", "numpy", "pyproj",
            "jupyterlab", "ipykernel", "nbclient", "nbformat"]
software = {"Python": platform.python_version(), **{name: version(name) for name in packages}}
display(pd.DataFrame.from_dict(software, orient="index", columns=["version"]))

## 4. Resolve and verify the canonical run

Resolution verifies the successful validation artifact, all declared run artifacts, exact file SHA-256, scientific-content signature, and topology signature before analysis begins.

In [ ]:
selection = resolve_canonical_run(
    ENVIRONMENT, canonical_run=CANONICAL_RUN, project_root=PROJECT_ROOT
)
print("Canonical run ID:", selection.run_id)
print("Canonical path:", selection.canonical_path.relative_to(PROJECT_ROOT).as_posix())
print("Manifest path:", selection.manifest_path.relative_to(PROJECT_ROOT).as_posix())
print("CANONICAL_RUN_RESOLUTION: PASS")

## 5. Concise canonical provenance

The table distinguishes exact file identity, scientific content, and connectivity identity. Git state describes the original canonical publication, not the current notebook session.

In [ ]:
manifest = selection.manifest
network_record = manifest["network_identity"]
git_record = manifest["git_provenance"]
provenance = pd.DataFrame({"value": {
    "canonical_run_id": selection.run_id,
    "file_sha256": selection.file_sha256,
    "scientific_content_signature": selection.scientific_content_signature,
    "topology_signature": selection.topology_signature,
    "git_commit": git_record["commit_sha"],
    "git_dirty_at_publication": git_record["dirty"],
    "crs": network_record["crs"],
    "node_count": network_record["node_count"],
    "physical_edge_count": network_record["physical_edge_count"],
}})
display(provenance)

## 6. Load the canonical GeoPackage and inspect its schema

The `nodes` and `edges` layers preserve stable scientific IDs and Shapely geometry. The GeoPackage—not the explanatory CSV exports—is authoritative.

In [ ]:
nodes, edges = load_canonical_geopackage(selection.canonical_path)
print(f"Loaded {len(nodes)} nodes and {len(edges)} physical edges")
display(nodes[["node_id", "x", "y", "degree", "geometry"]].head())
display(edges[["edge_id", "u", "v", "key", "length_local", "vertex_count", "geometry"]].head())

## 7. Recalculate geometry-derived values

Node coordinates and edge lengths are recalculated from actual geometry. Stored derived columns are comparison aids, never blindly trusted inputs.

In [ ]:
node_check = pd.DataFrame({
    "node_id": nodes.node_id,
    "stored_x": nodes.x,
    "geometry_x": nodes.geometry.x,
    "stored_y": nodes.y,
    "geometry_y": nodes.geometry.y,
})
edge_check = pd.DataFrame({
    "edge_id": edges.edge_id,
    "stored_length_local": edges.length_local,
    "geometry_length_local": edges.geometry.length,
})
assert np.allclose(node_check.stored_x, node_check.geometry_x)
assert np.allclose(node_check.stored_y, node_check.geometry_y)
assert np.allclose(edge_check.stored_length_local, edge_check.geometry_length_local)
display(node_check.head(3))
display(edge_check.head(3))
print("GEOMETRY_DERIVED_VALUES: PASS")

## 8. Validate canonical identity and geometry

The Phase 7 implementation checks required schema, Point/LineString validity, endpoints, derived values, CRS, scientific content, and topology.

In [ ]:
canonical = validate_canonical_for_analysis(nodes, edges)
assert sha256_file(selection.canonical_path) == selection.file_sha256
assert canonical.identity["scientific_content_signature"] == selection.scientific_content_signature
assert canonical.identity["topology_signature"] == selection.topology_signature
print("Canonical identity: PASS")
print("File SHA: PASS")
print("Scientific-content signature: PASS")
print("Topology signature: PASS")
print("CANONICAL_IDENTITY: PASS")

## 9. Build the NetworkX scientific graph

This undirected `MultiGraph` has exactly one edge per physical canonical street. The handshake theorem is demonstrated as an integrity check, not as a morphology metric.

In [ ]:
G_physical = build_networkx_multigraph(canonical, selection)
component_count = nx.number_connected_components(G_physical)
degree_sum = sum(dict(G_physical.degree()).values())
twice_edges = 2 * G_physical.number_of_edges()
assert degree_sum == twice_edges
print("Type:", type(G_physical).__name__)
print("Nodes:", G_physical.number_of_nodes())
print("Physical edges:", G_physical.number_of_edges())
print("Components:", component_count)
print(f"Handshake theorem: {degree_sum} = {twice_edges}")
print("NETWORKX_VALIDATION: 46 nodes | 69 physical edges | 1 component | PASS")

## 10. Build the OSMnx-compatible graph

The public OSMnx custom-GeoDataFrame workflow creates a reciprocal `MultiDiGraph`. Directed arc count and unique physical street count remain explicitly separate.

In [ ]:
G_osmnx = build_osmnx_multidigraph(canonical, selection)
correspondence = validate_graph_correspondence(canonical, G_physical, G_osmnx, selection)
print("Type:", type(G_osmnx).__name__)
print("Nodes:", G_osmnx.number_of_nodes())
print("Directed arcs:", directed_arc_count(G_osmnx))
print("Unique physical canonical edge IDs:", physical_street_count(G_osmnx))
print("OSMNX_VALIDATION: 46 nodes | 138 directed arcs | 69 physical canonical streets | PASS")

## 11. Demonstrate one physical edge and its reciprocal arcs

The first canonical edge is selected programmatically. Both arcs retain its canonical ID; endpoints are opposite, and reverse geometry is an exact coordinate reversal.

In [ ]:
example_edge_id = sorted(canonical.edges.edge_id)[0]
canonical_edge = canonical.edges.set_index("edge_id").loc[example_edge_id]
arc_records = [
    {"arc_u": u, "arc_v": v, "key": key, **data}
    for u, v, key, data in G_osmnx.edges(keys=True, data=True)
    if data["canonical_edge_id"] == example_edge_id
]
arc_table = pd.DataFrame([{
    "canonical_edge_id": record["canonical_edge_id"],
    "arc_direction": record["arc_direction"],
    "arc_u": record["arc_u"],
    "arc_v": record["arc_v"],
    "length_local": record["length_local"],
} for record in arc_records])
forward = next(record for record in arc_records if record["arc_direction"] == "forward")
reverse = next(record for record in arc_records if record["arc_direction"] == "reverse")
assert (forward["arc_u"], forward["arc_v"]) == (reverse["arc_v"], reverse["arc_u"])
assert tuple(reverse["geometry"].coords) == tuple(reversed(tuple(forward["geometry"].coords)))
display(pd.DataFrame([canonical_edge[["u", "v", "key", "length_local", "geometry"]]]))
display(arc_table)
print("RECIPROCAL_EDGE_DEMONSTRATION: PASS")

## 12. Visualize the canonical physical network

The figure uses authoritative physical LineStrings and an equal Cartesian aspect ratio. It deliberately avoids duplicating reciprocal directed arcs visually.

In [ ]:
svg = canonical_network_svg(
    canonical.nodes, canonical.edges, title="Env39 canonical physical street network"
)
display(SVG(svg))
print("PHYSICAL_NETWORK_VISUALIZATION: PASS")

## 13. Load and validate the Phase 7D GraphML artifact

GraphML is the portable directed representation. Reloading uses the existing typed OSMnx loader wrapper and repeats strict graph correspondence, geometry-orientation, numeric-attribute, metadata, and CRS checks.

In [ ]:
analysis_manifest_path = (
    PROJECT_ROOT / "results" / "analysis" / ENVIRONMENT / selection.run_id
    / "analysis_graph_manifest.json"
)
if not analysis_manifest_path.is_file():
    raise FileNotFoundError(
        f"Phase 7D analysis manifest is missing: {analysis_manifest_path}. "
        "Run scripts/build_analysis_graphs.py first."
    )
analysis_manifest = json.loads(analysis_manifest_path.read_text(encoding="utf-8"))
graphml_path = PROJECT_ROOT / analysis_manifest["artifacts"]["graphml_path"]
assert sha256_file(graphml_path) == analysis_manifest["artifacts"]["graphml_sha256"]
G_roundtrip = load_analysis_graphml(graphml_path)
roundtrip = validate_graphml_roundtrip(canonical, G_physical, G_roundtrip, selection)
print("GraphML path:", graphml_path.relative_to(PROJECT_ROOT).as_posix())
print("GraphML round-trip validation: PASS")
print("GRAPHML_ROUNDTRIP: 46 nodes | 138 directed arcs | 69 physical IDs | PASS")

## 14. Final validation summary

This notebook establishes the reproducible graph boundary. It does not calculate morphology metrics; those remain reserved for later phases.

In [ ]:
summary = f"""ENV39 CANONICAL DATA: PASS
Canonical run: {selection.run_id}

NETWORKX:
{G_physical.number_of_nodes()} nodes
{G_physical.number_of_edges()} physical edges
{component_count} component

OSMNX:
{G_osmnx.number_of_nodes()} nodes
{directed_arc_count(G_osmnx)} directed arcs
{physical_street_count(G_osmnx)} physical canonical streets

Coordinate system: GeoGami Local Cartesian
Units: local units
Ready for morphology metrics: YES
NOTEBOOK_01_EXECUTION: PASS"""
print(summary)